# CUB research suite v3 — one batch, explicit decisions

**Run All on Kaggle T4.** Default: full 200 classes, **11 experiments × 3 seeds**, with shared frozen DINOv2 features and shared representation training where appropriate. No improvement is pre-filled or promised. New results go to `/kaggle/working/relational_cub_suite_v3/`.

The v2 run showed (single seed): independent accuracy 79.63%; detached visibility 78.22%; detached graph 78.39%. Graph improved PCK by 0.87 percentage point over detached no-graph. Filtering reduced missing emissions but also reduced accuracy. The independent model's auxiliary visibility head had the best observed AUROC/Brier. **Our next hypothesis is that learning the classifier after the part detector and visibility are frozen can reduce this conflict.**

### Run instructions
1. Attach the same CUB dataset; select T4. Keep `REPAIR_LEGACY_CUDA=False`.
2. If possible attach the previous saved notebook Output containing `cache/`. Set `CACHE_INPUT` to that directory. In the same running session, the old cache is discovered automatically. Downloaded `.ipynb` files do **not** contain the large feature cache.
3. Run All. `PROFILE="smoke"` executes the entire suite on a small class subset with one seed first. `PROFILE="full"` is the default batch. Checkpoints allow same-configuration resume after a session ends.
4. Download **`decision_bundle.zip`** and return it for analysis. Save the whole Kaggle notebook Output to preserve trained models and feature caches.

### The batch (all matched part models use the same annotation supervision)
| ID | Purpose |
|---|---|
| `mean_linear` | Class-label-only linear classifier on masked mean DINO patch features; not a CLS-token or SOTA baseline |
| `independent` | Strong part-supervised model, no visibility gate |
| `posthoc_soft` | Apply visibility to the trained independent model without refitting; isolates inference mismatch |
| `staged_soft` | Freeze detector/visibility, refit class weights + temperature using soft evidence |
| `staged_normalized` | Same, normalize by available weighted evidence |
| `graph_ungated` | Anatomical graph without visibility in class scoring |
| `graph_normalized` | Anatomical multimodal graph + staged normalized evidence |
| `single_graph_normalized` | One Gaussian per edge instead of a mixture |
| `permuted_graph_normalized` | Wrong edge-to-prior correspondence; same graph and parameter count |
| `distance_graph_normalized` | Generic distance preference; no anatomical direction |
| `entropy_graph_normalized` | Anatomical graph with entropy-weighted neighbor messages |

All part models first train with **ungated** classification + localization + visibility BCE. That representation is frozen; phase 2 trains only class part-weights and temperature. Same-geometry variants share phase-1 training exactly. Posthoc uses the final independent classifier without phase-2 refitting under gates. Geometry variants require their own phase-1 training to be fair. No expensive new backbone or diffusion training is added.

### Evaluation boundaries
Official train/test split remains. The original 900-image validation subset is now split into 450 **selection** and 450 **calibration** examples. Training banks/geometry use training only; checkpoints use selection only; thresholds use calibration only. Default test reporting is enabled for this explicitly exploratory batch. Earlier CUB test results informed development, so this is **not an untouched confirmatory benchmark**. Lock the winning recipe before a fresh external dataset or reserved future evaluation.

A 2D geometry cluster, image rotation, or synthetic occlusion is not proof of novel-view/3D understanding. Exact published baselines and a second dataset remain later work; see the accompanying full research pipeline.


In [ ]:
# 1. Configuration — this is the main cell to edit.
from pathlib import Path
PROFILE = "full"                 # "smoke", "standard", "research"
DATA_ROOT = Path("/kaggle/input/datasets/wenewone/cub2002011")
OUT = Path("/kaggle/working/relational_cub_suite_v3")
OUT.mkdir(parents=True, exist_ok=True)

# Optional: attach a previous notebook output and point to its cache/checkpoint directory.
CACHE_INPUT = next((str(p) for p in [Path("/kaggle/working/relational_cub_v2/cache"), Path("/kaggle/working/relational_cub/cache")] if p.exists()), None)                    # e.g. "/kaggle/input/my-run/relational_cub/cache"
RESUME_INPUT = None                   # e.g. "/kaggle/input/my-run/relational_cub/runs"
# Offline DINOv2: provide BOTH a source folder containing hubconf.py and a .pth weight file.
DINO_SOURCE = None
DINO_WEIGHTS = None

PROFILES = {
    "smoke": dict(max_classes=10, epochs=2, seeds=[42], robustness_n=16),
    "full": dict(max_classes=None, epochs=12, seeds=[42,123,2026], robustness_n=256),
}
assert PROFILE in PROFILES
CFG = dict(
    **PROFILES[PROFILE], profile=PROFILE, image_size=224, patch_size=14, dim=384,
    extraction_batch=24, batch_size=48, workers=2, lr=1e-3, weight_decay=1e-4,
    val_fraction=0.15, split_seed=1729, prototype_seed=2026,
    class_part_modes=2, graph_modes=4, graph_steps=2,
    loc_temperature=0.12, loc_sigma_patches=0.8, relation_strength_max=1.5,
    lambda_loc=0.5, lambda_visibility=0.5, lambda_edge=0.0,
    threshold_visible_recall=0.90, threshold_min_examples=10,
    prototype_k=5, prototype_batches=2, prototype_momentum=0.9,
    pck_fraction=0.10, visibility_threshold=0.5, occlusion_side=0.18,
    backbone="dinov2_vits14_reg", dino_commit="e1277af2ba9496fbadf7aec6eba56e8d882d1e35",
)
EXPERIMENTS = ["parts_independent", "parts_visibility",
               "parts_visibility_detached", "parts_graph_detached"]
# Optional matched graph ablation: append "parts_graph_single_detached".
# Optional historical joint-visibility graph: append "parts_graph".
# Optional weak PNP reference: append "pnp_frozen" (not an exact paper reproduction).
RUN_ROBUSTNESS = True
AUTO_RESUME = True
REPAIR_LEGACY_CUDA = False  # Only for an incompatible P100/T4 environment; see Environment below.
# Reusing runs with a different method/configuration is rejected automatically.

HEAD_EPOCHS = 2 if PROFILE=="smoke" else 15
LINEAR_EPOCHS = 3 if PROFILE=="smoke" else 30
EVALUATE_TEST = True  # False: selection-only developmental reporting. No test metrics then.
CALIBRATION_RECALL_GRID = [0.80,0.90,0.95,0.98]
PRIMARY_RECALL = 0.90
OCCLUSION_SIDES = [0.08,0.14,0.20]
# Set to an attached old output's `occlusion` folder to reuse EXACT matching old corruption caches,
# or let this suite generate its own severity-specific caches. No old model checkpoint is reused.
SUITE = {
 "mean_linear": dict(geometry="none",score="linear"),
 "independent": dict(geometry="none",score="none"),
 "posthoc_soft": dict(geometry="none",score="soft",posthoc=True),
 "staged_soft": dict(geometry="none",score="soft"),
 "staged_normalized": dict(geometry="none",score="normalized"),
 "graph_ungated": dict(geometry="anatomy",score="none"),
 "graph_normalized": dict(geometry="anatomy",score="normalized"),
 "single_graph_normalized": dict(geometry="single",score="normalized"),
 "permuted_graph_normalized": dict(geometry="permuted",score="normalized"),
 "distance_graph_normalized": dict(geometry="distance",score="normalized"),
 "entropy_graph_normalized": dict(geometry="entropy",score="normalized"),
}
RUN_VARIANTS = list(SUITE)  # Run the full batch by default; preserve order for posthoc control.
# EXPERIMENTS below is only the small legacy API test list, not the research batch.
EXPERIMENTS = ["parts_independent","parts_visibility","parts_visibility_detached","parts_graph_detached","parts_graph_single_detached"]


## Environment
Kaggle normally includes these packages. The default keeps the installed PyTorch stack and runs GPU kernel probes **before data preprocessing**. GPU detection alone does not establish binary compatibility.

If you see `no kernel image is available for execution on the device`, restart the kernel first. Inspect the printed GPU name, compute capability, PyTorch CUDA version and compiled architectures. Pascal/P100 is not supported by newer CUDA 12.8+/13.x PyTorch builds. For a P100 or T4 environment with this error, set `REPAIR_LEGACY_CUDA=True`, run Configuration and Environment once with Internet on, then **restart the kernel, set the flag back to False, and Run All**. The optional repair installs the official matched PyTorch 2.7.1 / torchvision 0.22.1 / torchaudio 2.7.1 CUDA 12.6 wheels. Do not use that legacy repair for a Blackwell GPU. Switching to an available T4 accelerator is another option; still run the probes.

Installation changes files on disk; a restart is required to replace already loaded CUDA libraries. Completed feature batches are retained. Sources: [official version matrix](https://pytorch.org/get-started/previous-versions/), [PyTorch architecture-support notice](https://dev-discuss.pytorch.org/t/cuda-toolkit-version-and-architecture-support-update-maxwell-and-pascal-architecture-support-removed-in-cuda-12-8-and-12-9-builds/3128).

No packages are replaced automatically. A missing package is installed individually. DINOv2 code is pinned to the commit referenced by the supplied implementation. This notebook uses standard attention with xFormers disabled.

In [ ]:
import os, sys, json, time, math, random, hashlib, gc, shutil, tarfile, zipfile
import importlib.util, subprocess, platform, copy, warnings
os.environ["XFORMERS_DISABLED"] = "1"
if REPAIR_LEGACY_CUDA:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir",
        "torch==2.7.1+cu126", "torchvision==0.22.1+cu126", "torchaudio==2.7.1+cu126",
        "--index-url", "https://download.pytorch.org/whl/cu126"])
    raise RuntimeError("Installation finished. RESTART the kernel now, set REPAIR_LEGACY_CUDA=False, then Run All. Do not continue with the old imported CUDA libraries.")
for module, package in [("sklearn", "scikit-learn"), ("matplotlib", "matplotlib"),
                        ("pandas", "pandas"), ("PIL", "Pillow"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    warnings.warn("No GPU selected. Enable a Kaggle GPU before a full run; CPU extraction is slow.")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
# Probe real kernels, not only torch.cuda.is_available(). Failed CUDA contexts need a restart.
def cuda_preflight():
    info=dict(torch=str(torch.__version__), torch_cuda=torch.version.cuda,
              cuda_available=torch.cuda.is_available())
    if DEVICE.type != "cuda":
        return False, info
    info.update(gpu=torch.cuda.get_device_name(0), capability=list(torch.cuda.get_device_capability(0)),
                compiled_architectures=torch.cuda.get_arch_list(), cudnn=torch.backends.cudnn.version())
    print("GPU compatibility report:", json.dumps(info,indent=2))
    def probe(amp):
        with torch.inference_mode(), torch.autocast("cuda",dtype=torch.float16,enabled=amp):
            # Same 14x14 stride-14 convolution as DINO's patch embedding, plus attention-like ops.
            x=torch.randn(2,3,56,56,device=DEVICE)
            w=torch.randn(24,3,14,14,device=DEVICE)
            y=F.conv2d(x,w,stride=14).flatten(2).transpose(1,2)
            z=((y @ y.transpose(-1,-2))/math.sqrt(24)).softmax(-1) @ y
            result=F.normalize(z.float(),dim=-1)
            torch.cuda.synchronize()
            if not torch.isfinite(result).all().item(): raise FloatingPointError("Non-finite GPU probe")
    try:
        probe(False)
    except RuntimeError as exc:
        raise RuntimeError("GPU FP32 kernel probe failed BEFORE feature extraction. Restart the kernel. "
            "Check the compatibility report above; for an incompatible P100/T4 wheel, use the optional "
            "REPAIR_LEGACY_CUDA step documented above, or switch to a supported GPU. "
            "Batch size, dataset annotations, and re-downloading DINO weights do not repair a missing GPU kernel. "
            f"Original error: {exc}") from exc
    try:
        probe(True)
    except RuntimeError as exc:
        raise RuntimeError("FP32 passed but FP16 kernel probe failed. Restart the kernel and use a compatible "
            f"PyTorch/CUDA build; share the compatibility report if unsure. Original error: {exc}") from exc
    print("PASS: GPU convolution, matrix multiplication and normalization in FP32 and FP16.")
    return True, info

AMP_ENABLED, CUDA_REPORT = cuda_preflight()
save_cuda_report_path = OUT/"cuda_compatibility.json"
save_cuda_report_path.write_text(json.dumps(CUDA_REPORT,indent=2,default=str))
# One GPU is used intentionally: cached feature heads do not need distributed training.
torch.set_num_threads(min(4, os.cpu_count() or 1))
MEAN = np.array([0.485,0.456,0.406], dtype=np.float32)
STD = np.array([0.229,0.224,0.225], dtype=np.float32)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]

def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str)); tmp.replace(path)

def atomic_torch_save(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp"); torch.save(obj, tmp); tmp.replace(path)

def load_local_checkpoint(path):
    # Only load checkpoints produced by this notebook or ones you trust.
    return torch.load(path, map_location="cpu", weights_only=False)

VERSIONS = dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__,
                pandas=pd.__version__, device=str(DEVICE))
print(VERSIONS)
seed_all(CFG["prototype_seed"])
save_json(dict(config=CFG, experiments=EXPERIMENTS, versions=VERSIONS), OUT/"configuration.json")

## Data and leakage controls
Official test IDs remain test IDs. A stratified validation split is taken **only from official training IDs**. Prototype banks, PCA foreground selections, geometry mixtures, and class-part availability use training IDs only. Validation chooses checkpoints; test never drives early stopping or thresholds. No ground-truth box crop or part coordinate is passed into the model at inference.

Images are resized with preserved aspect ratio and centered padding. Keypoints and bounding boxes use the exact same transform; padding tokens are excluded from localization. No offline 40× augmentation or ViT block expansion is performed: these are deliberate departures from the original paper.

In [ ]:
# 3. Find the annotation root, including nested Kaggle layouts and local archives.
REQUIRED = ["images.txt", "image_class_labels.txt", "train_test_split.txt", "bounding_boxes.txt", "parts/part_locs.txt"]
def find_cub(root):
    root = Path(root)
    candidates = [root] + sorted({p.parent for p in root.rglob("images.txt")}) if root.exists() else []
    for p in candidates:
        if all((p/q).is_file() for q in REQUIRED) and (p/"images").is_dir(): return p
    return None

def extract_local_archive(archive, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    def valid(name):
        target = (dest/name).resolve()
        return target.is_relative_to(dest.resolve())
    if tarfile.is_tarfile(archive):
        with tarfile.open(archive) as ar:
            members = ar.getmembers()
            if any(not valid(m.name) or m.issym() or m.islnk() or m.isdev() for m in members):
                raise ValueError(f"Unsafe archive member in {archive}")
            ar.extractall(dest, members=members)
    elif zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as ar:
            if any(not valid(m.filename) or ((m.external_attr >> 16) & 0o170000) == 0o120000 for m in ar.infolist()):
                raise ValueError(f"Unsafe archive member in {archive}")
            ar.extractall(dest)

CUB_ROOT = find_cub(DATA_ROOT)
if CUB_ROOT is None:
    # Also accept Kaggle's older mount convention.
    alternate = Path("/kaggle/input/cub2002011")
    if alternate.exists():
        DATA_ROOT = alternate; CUB_ROOT = find_cub(DATA_ROOT)
if CUB_ROOT is None and DATA_ROOT.exists():
    extracted = OUT/"extracted_cub"
    CUB_ROOT = find_cub(extracted)
    if CUB_ROOT is None:
        archives = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and
                          p.name.lower().endswith((".tgz", ".tar.gz", ".tar", ".zip")))
        for ar in archives:
            print("Extracting:", ar.name)
            extract_local_archive(ar, extracted)
            CUB_ROOT = find_cub(extracted)
            if CUB_ROOT is not None: break
if CUB_ROOT is None:
    raise FileNotFoundError(f"CUB images and standard annotation files not found under {DATA_ROOT}. Attach the complete dataset or edit DATA_ROOT.")
print("CUB root:", CUB_ROOT)

def read_table(relative, names):
    return pd.read_csv(CUB_ROOT/relative, sep=r"\s+", header=None, names=names)

frame = read_table("images.txt", ["id","relative_path"])
for file, names in [("image_class_labels.txt", ["id","original_class"]),
                    ("train_test_split.txt", ["id","official_train"]),
                    ("bounding_boxes.txt", ["id","box_x","box_y","box_w","box_h"])]:
    frame = frame.merge(read_table(file,names), on="id", validate="one_to_one")
frame = frame.sort_values("id").reset_index(drop=True)
selected_classes = sorted(frame.original_class.unique())
if CFG["max_classes"] is not None: selected_classes = selected_classes[:CFG["max_classes"]]
frame = frame[frame.original_class.isin(selected_classes)].reset_index(drop=True)
class_map = {int(c):i for i,c in enumerate(selected_classes)}
frame["label"] = frame.original_class.map(class_map)
frame["path"] = frame.relative_path.map(lambda p: str(CUB_ROOT/"images"/p))
assert all(Path(p).is_file() for p in frame.path), "Missing image files"
assert set(frame.official_train.unique()) == {0,1}
class_names = [frame.loc[frame.original_class.eq(c),"relative_path"].iloc[0].split("/")[0] for c in selected_classes]
part_names = ["back","beak","belly","breast","crown","forehead","left eye","left leg",
              "left wing","nape","right eye","right leg","right wing","tail","throat"]
if (CUB_ROOT/"parts/parts.txt").exists():
    with open(CUB_ROOT/"parts/parts.txt") as f:
        lookup = {int(line.split(maxsplit=1)[0]):line.strip().split(maxsplit=1)[1] for line in f if line.strip()}
    assert sorted(lookup) == list(range(1,16))
    part_names = [lookup[i] for i in range(1,16)]
locs = read_table("parts/part_locs.txt", ["id","part","x","y","visible"])
locs = locs[locs.id.isin(frame.id)]
assert not locs.duplicated(["id","part"]).any()
assert locs.groupby("id").size().eq(15).all() and set(locs.id) == set(frame.id)
assert set(locs.part.unique()) == set(range(1,16))
locs = locs.set_index(["id","part"]).reindex(pd.MultiIndex.from_product([frame.id,range(1,16)],names=["id","part"]))
raw_points = locs[["x","y"]].to_numpy(np.float32).reshape(-1,15,2)
raw_visible = locs.visible.to_numpy(np.float32).reshape(-1,15)
assert np.isfinite(raw_points).all() and np.isin(raw_visible,[0,1]).all()
all_train = np.flatnonzero(frame.official_train.to_numpy() == 1)
train_idx, val_idx = train_test_split(all_train, test_size=CFG["val_fraction"],
    random_state=CFG["split_seed"], stratify=frame.label.to_numpy()[all_train])
test_idx = np.flatnonzero(frame.official_train.to_numpy() == 0)
train_idx, val_idx, test_idx = map(np.sort, [train_idx,val_idx,test_idx])
assert not (set(train_idx)&set(val_idx) or set(train_idx)&set(test_idx) or set(val_idx)&set(test_idx))
C, P = len(selected_classes), 15
N_GRID = CFG["image_size"]//CFG["patch_size"]
assert CFG["image_size"] % CFG["patch_size"] == 0
T = N_GRID**2
split_manifest = {name: frame.iloc[idx].id.tolist() for name,idx in [("train",train_idx),("val",val_idx),("test",test_idx)]}
save_json(dict(ids=split_manifest,class_names=class_names,part_names=part_names), OUT/"splits.json")
print(f"Classes={C}; train={len(train_idx)}, validation={len(val_idx)}, test={len(test_idx)}")
if C != 200: print("SMOKE SUBSET: these results are not full CUB-200 results.")
display(pd.DataFrame({"part_id":np.arange(1,16),"part":part_names}))

selection_idx,calibration_idx=train_test_split(val_idx,test_size=.5,random_state=2718,
    stratify=frame.label.to_numpy()[val_idx])
selection_idx,calibration_idx=np.sort(selection_idx),np.sort(calibration_idx)
REPORT_ROWS=test_idx if EVALUATE_TEST else selection_idx
REPORT_SPLIT="test_development" if EVALUATE_TEST else "selection_development"
assert not (set(selection_idx)&set(calibration_idx))
split_manifest.update(selection=frame.iloc[selection_idx].id.tolist(),calibration=frame.iloc[calibration_idx].id.tolist())
save_json(dict(ids=split_manifest,class_names=class_names,part_names=part_names),OUT/"splits.json")
print("Model selection:",len(selection_idx),"Threshold calibration:",len(calibration_idx),"Report:",REPORT_SPLIT)


In [ ]:
# 4. Transform images and annotations without dropping any part IDs.
def letterbox(index, return_pil=False):
    row = frame.iloc[int(index)]
    im = Image.open(row.path).convert("RGB")
    w,h = im.size; size = CFG["image_size"]
    scale = size/max(w,h)
    nw,nh = max(1,round(w*scale)), max(1,round(h*scale))
    ox,oy = (size-nw)//2, (size-nh)//2
    sx,sy = nw/w, nh/h
    canvas = Image.new("RGB",(size,size),tuple(int(v*255) for v in MEAN))
    canvas.paste(im.resize((nw,nh),Image.Resampling.BILINEAR),(ox,oy))
    # CUB image coordinates are treated as one-based; normalized coordinates use pixel centers.
    points = ((raw_points[index]-1)*np.array([sx,sy])+np.array([ox+0.5,oy+0.5]))/size
    visible = raw_visible[index].copy()
    in_bounds = ((raw_points[index,:,0]>=1)&(raw_points[index,:,0]<=w)&
                 (raw_points[index,:,1]>=1)&(raw_points[index,:,1]<=h))
    visible *= in_bounds
    points = np.clip(points,0,1).astype(np.float32)
    box = np.array([(row.box_x-1)*sx+ox,(row.box_y-1)*sy+oy,row.box_w*sx,row.box_h*sy],np.float32)/size
    gy,gx = np.meshgrid((np.arange(N_GRID)+0.5)/N_GRID,(np.arange(N_GRID)+0.5)/N_GRID,indexing="ij")
    grid = np.stack([gx,gy],-1).reshape(-1,2)
    valid = ((grid[:,0]>=ox/size)&(grid[:,0]<(ox+nw)/size)&
             (grid[:,1]>=oy/size)&(grid[:,1]<(oy+nh)/size))
    if not valid.any(): valid[np.argmin(((grid-.5)**2).sum(1))]=True
    tensor = torch.from_numpy(((np.asarray(canvas).astype(np.float32)/255-MEAN)/STD).transpose(2,0,1).copy())
    meta = dict(points=points,visible=visible,box=box,valid=valid)
    return (canvas,meta) if return_pil else (tensor,meta)

# Store small metadata separately from the large feature array.
points=np.zeros((len(frame),P,2),np.float32); visibility=np.zeros((len(frame),P),np.float32)
boxes=np.zeros((len(frame),4),np.float32); valid_tokens=np.zeros((len(frame),T),bool)
for i in tqdm(range(len(frame)),desc="Image geometry"):
    _,m=letterbox(i)
    points[i],visibility[i],boxes[i],valid_tokens[i]=m["points"],m["visible"],m["box"],m["valid"]
GRID = torch.tensor(np.stack(np.meshgrid((np.arange(N_GRID)+.5)/N_GRID,
    (np.arange(N_GRID)+.5)/N_GRID,indexing="xy"),-1).reshape(-1,2),dtype=torch.float32)

fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,i in zip(axes,train_idx[:3]):
    im,_=letterbox(int(i),True); ax.imshow(im)
    for p in np.flatnonzero(visibility[i]):
        x,y=points[i,p]*CFG["image_size"]; ax.scatter(x,y,s=10); ax.text(x,y,str(p+1),fontsize=7)
    ax.set_title(class_names[int(frame.iloc[i].label)],fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(OUT/"annotation_alignment.png",dpi=140); plt.show()
print("Check the annotated points above before trusting localization metrics.")

## Frozen DINOv2 feature cache
The cache is float16 on disk and read by memory mapping. At 224px, all 11,788 images need approximately 2.16 GiB for 384-dimensional patch features. Padding masks are retained. Cache fingerprints include annotation content, IDs, backbone, commit, image size and preprocessing version. Completed extraction batches are resumable. On a new Kaggle session, attach your previous output and set `CACHE_INPUT`.

In [ ]:
# 5. DINOv2 loader: pinned official code, explicit local/offline mode.
def load_backbone():
    if DINO_SOURCE is not None:
        if DINO_WEIGHTS is None: raise ValueError("Set DINO_WEIGHTS along with DINO_SOURCE.")
        model = torch.hub.load(str(DINO_SOURCE),CFG["backbone"],source="local",pretrained=False)
        state = torch.load(DINO_WEIGHTS,map_location="cpu",weights_only=True)
        if "state_dict" in state: state=state["state_dict"]
        model.load_state_dict(state,strict=True)
    else:
        try:
            model = torch.hub.load(f"facebookresearch/dinov2:{CFG['dino_commit']}",
                                  CFG["backbone"],pretrained=True,trust_repo=True,skip_validation=True)
        except Exception as exc:
            raise RuntimeError("DINOv2 download failed. Enable Kaggle Internet, or attach official source and weights and set DINO_SOURCE/DINO_WEIGHTS. No random-weight fallback is used.") from exc
    model.eval().requires_grad_(False)
    return model.to(DEVICE)

class ImageRows(Dataset):
    def __init__(self,rows): self.rows=list(map(int,rows))
    def __len__(self): return len(self.rows)
    def __getitem__(self,j):
        i=self.rows[j]; x,_=letterbox(i)
        return x,i

annotation_hashes={f:hashlib.sha256((CUB_ROOT/f).read_bytes()).hexdigest() for f in REQUIRED}
cache_spec=dict(version="letterbox-v3",ids=frame.id.tolist(),annotations=annotation_hashes,
                image_size=CFG["image_size"],backbone=CFG["backbone"],commit=CFG["dino_commit"],
                local_weight_hash=(hashlib.sha256(Path(DINO_WEIGHTS).read_bytes()).hexdigest() if DINO_WEIGHTS else None))
CACHE_KEY=stable_hash(cache_spec)
cache_dir=OUT/"cache"; cache_dir.mkdir(exist_ok=True)
cache_file=cache_dir/f"features_{CACHE_KEY}.npy"
done_file=cache_dir/f"done_{CACHE_KEY}.npy"
if CACHE_INPUT:
    source=Path(CACHE_INPUT)
    for target in [cache_file,done_file]:
        if not target.exists() and (source/target.name).exists(): shutil.copy2(source/target.name,target)
shape=(len(frame),T,CFG["dim"])
if cache_file.exists():
    features=np.load(cache_file,mmap_mode="r+")
    assert features.shape==shape and features.dtype==np.float16, "Incompatible feature cache"
    done=np.load(done_file) if done_file.exists() else np.zeros(len(frame),bool)
else:
    need=np.prod(shape)*2
    if shutil.disk_usage(OUT).free<need*1.15: raise RuntimeError("Insufficient disk space for feature cache")
    features=np.lib.format.open_memmap(cache_file,mode="w+",dtype=np.float16,shape=shape)
    done=np.zeros(len(frame),bool)
missing=np.flatnonzero(~done)
print(f"Feature cache: {int(done.sum())} complete, {len(missing)} images to extract.")
if len(missing):
    backbone=load_backbone()
    loader=DataLoader(ImageRows(missing),batch_size=CFG["extraction_batch"],shuffle=False,
                      num_workers=CFG["workers"],pin_memory=DEVICE.type=="cuda")
    start=time.time()
    with torch.inference_mode():
        for images,rows in tqdm(loader,desc="DINOv2 cache"):
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=AMP_ENABLED):
                f=backbone.forward_features(images.to(DEVICE,non_blocking=True))["x_norm_patchtokens"]
            assert tuple(f.shape[1:])==(T,CFG["dim"])
            f=F.normalize(f.float(),dim=-1)
            if not torch.isfinite(f).all(): raise FloatingPointError("Non-finite DINO features")
            features[rows.numpy()]=f.cpu().numpy().astype(np.float16)
            features.flush(); done[rows.numpy()]=True
            tmp=done_file.with_suffix(".tmp.npy"); np.save(tmp,done); tmp.replace(done_file)
    save_json(dict(seconds=time.time()-start,images=len(missing)),OUT/"extraction_timing.json")
    del backbone; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
assert done.all()
features=np.load(cache_file,mmap_mode="r")
save_json(cache_spec,cache_dir/f"manifest_{CACHE_KEY}.json")
print("Feature cache ready:",features.shape,"GiB",round(features.nbytes/2**30,2))

## Mathematical hypotheses, each with a control

### A. Separate evidence learning from class scoring
Phase 1 uses $\mathcal L_{CE}^{ungated}+\lambda_{loc}\mathcal L_{heatmap}+\lambda_{vis}\mathcal L_{BCE}$.
After checkpoint selection, freeze the part detector, visibility head, prototypes and graph strength. Phase 2 minimizes classification CE over **class part-weights and a bounded scalar temperature only**. The frozen gate no longer drifts while the classifier learns its operating distribution. Every staged part model receives a matched ungated representation-learning phase; this is not the v2 detached method relabeled.

### B. Normalize by available evidence (experimental, not assumed correct)
Let $e_{cp}$ be class/part cosine evidence, $v_p$ frozen predicted visibility, and $w_{cp}$ nonnegative class part-weights summing to one.

$$z_c^{soft}=\tau\sum_p w_{cp}v_pe_{cp},\qquad
z_c^{norm}=\tau\frac{\sum_p w_{cp}v_pe_{cp}}{\epsilon+\sum_p w_{cp}v_p}.$$

The implementation clamps the denominator to epsilon. Normalization can reduce changes in logit scale caused by missing parts, but may also overvalue a few weak surviving parts; inspect accuracy and coverage. Class-dependent denominators introduce their own bias and must be evaluated, not presented as a theorem. Contributions are divided by the same denominator and still sum to the logit. All-zero evidence yields abstention.

### C. Test anatomical information against generic regularization
Multimodal $K_{pj}$ is fitted to training relative 2D coordinates. Single-mode, mismatched-edge and isotropic-distance kernels are controls. All have the same trainable detector and learned bounded message strength.

$$m_{pt}=\frac1{|N(p)|}\sum_{j\in N(p)}v_j\log\left(\epsilon+\sum_s K_{pj}(t,s)q_{js}\right).$$

The entropy variant alone replaces $v_j$ by $v_j(1-H(q_j)/\log N_{valid})$, with detached confidence. This is a separate uncertainty-weighting hypothesis. Kernel controls and seeds must support an anatomy-specific claim; "adding a graph" is not sufficient novelty.

### D. Rejection is a policy, not proof of better representation
For each predeclared recall target $r\in\{.80,.90,.95,.98\}$, fit part thresholds on **calibration only** and filter actual class contributions. The primary operating point is fixed at .90. Raw and filtered results remain separate. No best threshold is selected using test results.

Graph context and DINO tokens may retain information from rejected parts. Removing an explicit contribution is not causal erasure of all indirect influence.


In [ ]:
# 6. Training-only semantic banks and anatomical compatibility kernels.
def heatmap_targets(xy,valid,sigma=None):
    grid=GRID.to(xy.device)
    sigma=sigma or CFG["loc_sigma_patches"]/N_GRID
    logq=-((xy[:,:,None,:]-grid[None,None,:,:])**2).sum(-1)/(2*sigma*sigma)
    return logq.masked_fill(~valid[:,None,:],-1e4).softmax(-1)

def normalize_np(x): return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-8)

def build_semantic_banks():
    descriptors=[]
    for start in range(0,len(train_idx),128):
        rows=train_idx[start:start+128]
        q=heatmap_targets(torch.tensor(points[rows]),torch.tensor(valid_tokens[rows])).numpy()
        f=np.asarray(features[rows],dtype=np.float32)
        descriptors.append(normalize_np(np.einsum("bpt,btd->bpd",q,f)))
    desc=np.concatenate(descriptors); labels=frame.label.to_numpy()[train_idx]
    vis=visibility[train_idx]>0
    anchors=np.zeros((P,CFG["dim"]),np.float32)
    for p in range(P):
        if not vis[:,p].any(): raise ValueError(f"No training observations for {part_names[p]}; use more classes.")
        anchors[p]=normalize_np(desc[vis[:,p],p].mean(0))
    bank=np.zeros((C,P,CFG["class_part_modes"],CFG["dim"]),np.float32)
    support=np.zeros(bank.shape[:-1],bool); counts=np.zeros((C,P),int)
    for c in tqdm(range(C),desc="Class/part banks"):
        for p in range(P):
            x=desc[(labels==c)&vis[:,p],p]; counts[c,p]=len(x)
            if not len(x): continue
            k=min(CFG["class_part_modes"],len(x))
            centers=KMeans(k,n_init=5,random_state=CFG["prototype_seed"]).fit(x).cluster_centers_ if k>1 else x.mean(0,keepdims=True)
            bank[c,p,:k]=normalize_np(centers); support[c,p,:k]=True
    assert support.any((1,2)).all()
    return torch.tensor(anchors),torch.tensor(bank),torch.tensor(support),counts

# CUB IDs minus one. This is an explicit anatomical prior, not claimed to be discovered.
UNDIRECTED=[(1,5),(5,4),(4,9),(9,14),(14,3),(3,2),(2,0),
            (0,8),(0,12),(2,7),(2,11),(0,13),(4,6),(4,10)]
EDGES=UNDIRECTED+[(j,i) for i,j in UNDIRECTED]

def build_kernels(modes):
    grid=GRID.numpy(); delta=grid[None,:,:]-grid[:,None,:] # neighbor minus candidate
    kernels=[]; fit_info=[]
    for p,j in EDGES:
        ok=(visibility[train_idx,p]>0)&(visibility[train_idx,j]>0)
        x=points[train_idx[ok],j]-points[train_idx[ok],p]
        k=min(modes,max(1,len(x)//10))
        if len(x)<3: raise ValueError(f"Too few observations for edge {part_names[p]} -> {part_names[j]}")
        gm=GaussianMixture(n_components=k,covariance_type="full",reg_covar=(0.7/N_GRID)**2,
                           n_init=2,random_state=CFG["prototype_seed"]).fit(x)
        logk=gm.score_samples(delta.reshape(-1,2)).reshape(T,T)
        kernel=np.exp(logk-logk.max()).astype(np.float32)
        kernels.append(kernel)
        fit_info.append(dict(source=part_names[p],neighbor=part_names[j],n=len(x),
                             weights=gm.weights_.tolist(),means=gm.means_.tolist(),covariances=gm.covariances_.tolist()))
    return torch.tensor(np.stack(kernels)),fit_info

anchors,part_bank,part_support,bank_counts=build_semantic_banks()
GRAPH_KERNELS,graph_info=build_kernels(CFG["graph_modes"])
SINGLE_KERNELS,single_info=build_kernels(1) if any(v in EXPERIMENTS for v in ["parts_graph_single", "parts_graph_single_detached"]) else (None,None)
save_json(graph_info,OUT/"geometry_mixtures.json")
pd.DataFrame(bank_counts,columns=part_names,index=class_names).to_csv(OUT/"training_part_counts.csv")
print("Unavailable class-part pairs:",int((bank_counts==0).sum()),"of",C*P)

In [ ]:
# 7. A source-inspired frozen-feature PNP reference.
# Adapted conceptually from modeling/pnp.py and modeling/utils.py in the supplied ZIP:
# class-wise Sinkhorn assignment, non-gradient prototype updates, max similarity and weighted aggregation.
# Differences: per-image PCA with a deterministic central-sign heuristic; normalized cluster means;
# no background prototype, no PPC/expanded ViT training, full-image letterbox, no 40x augmentation.
@torch.no_grad()
def balanced_assign(scores,iterations=5,epsilon=0.05):
    logq=(scores.float()/epsilon).T
    logq=logq-logq.max()
    q=logq.exp().clamp_min(1e-12); q=q/q.sum()
    k,n=q.shape
    for _ in range(iterations):
        q=q/q.sum(1,keepdim=True).clamp_min(1e-12)/k
        q=q/q.sum(0,keepdim=True).clamp_min(1e-12)/n
    return (q*n).T

def pca_foreground(f,valid):
    # Neither part points nor bounding boxes are used here.
    idx=np.flatnonzero(valid); x=f[idx].astype(np.float32); xc=x-x.mean(0)
    # Power iteration on XX^T avoids fitting a 384-D SVD per image.
    rng=np.random.default_rng(0); v=rng.normal(size=x.shape[1]).astype(np.float32)
    for _ in range(8):
        v=xc.T@(xc@v); v=v/(np.linalg.norm(v)+1e-8)
    score=xc@v; g=GRID.numpy()[idx]
    center=((g-.5)**2).sum(1)<0.18**2
    if center.any() and (~center).any() and score[center].mean()<score[~center].mean(): score=-score
    chosen=idx[score>=np.median(score)]
    return f[chosen]

def build_pnp_bank():
    rng=np.random.default_rng(CFG["prototype_seed"])
    bank=[]; labels=frame.label.to_numpy()
    for c in tqdm(range(C),desc="PNP foreground clustering"):
        chunks=[]
        for i in train_idx[labels[train_idx]==c]:
            f=np.asarray(features[i],dtype=np.float32)
            chunks.append(pca_foreground(f,valid_tokens[i]))
        x=torch.tensor(normalize_np(np.concatenate(chunks)))
        k=CFG["prototype_k"]
        proto=x[torch.tensor(rng.choice(len(x),k,replace=len(x)<k))].clone()
        for _ in range(CFG["prototype_batches"]):
            order=torch.tensor(rng.permutation(len(x)))
            for batch in order.split(1024):
                xb=x[batch]; a=balanced_assign(xb@F.normalize(proto,dim=-1).T)
                means=a.T@xb/a.sum(0)[:,None].clamp_min(1e-8)
                proto=F.normalize(CFG["prototype_momentum"]*proto+(1-CFG["prototype_momentum"])*means,dim=-1)
        bank.append(proto)
    return torch.stack(bank)

pnp_bank=build_pnp_bank() if "pnp_frozen" in EXPERIMENTS else None
atomic_torch_save(dict(anchors=anchors,part_bank=part_bank,part_support=part_support,
    graph=GRAPH_KERNELS,single=SINGLE_KERNELS,pnp=pnp_bank,edges=EDGES,grid=GRID,
    counts=bank_counts,cache_key=CACHE_KEY),OUT/"training_banks.pt")

In [ ]:
# 8. Datasets and trainable heads. Every forward() takes features + padding mask ONLY.
class FeatureRows(Dataset):
    def __init__(self,rows): self.rows=np.asarray(rows,dtype=int)
    def __len__(self): return len(self.rows)
    def __getitem__(self,j):
        i=int(self.rows[j])
        return dict(features=torch.from_numpy(np.array(features[i],dtype=np.float32)),
                    valid=torch.tensor(valid_tokens[i]),points=torch.tensor(points[i]),
                    visible=torch.tensor(visibility[i]),box=torch.tensor(boxes[i]),
                    label=torch.tensor(int(frame.iloc[i].label)),row=torch.tensor(i))

def move_batch(batch): return {k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}

def feature_loader(rows,shuffle=False,generator=None):
    return DataLoader(FeatureRows(rows),batch_size=CFG["batch_size"],shuffle=shuffle,
        num_workers=0,pin_memory=DEVICE.type=="cuda",generator=generator)

class FrozenPNP(nn.Module):
    def __init__(self):
        super().__init__(); self.register_buffer("bank",pnp_bank.clone())
        self.weights=nn.Parameter(torch.zeros(C,CFG["prototype_k"]))
    def forward(self,f,valid):
        sim=torch.einsum("btd,ckd->bckt",F.normalize(f,dim=-1),self.bank)
        sim=sim.masked_fill(~valid[:,None,None,:],-1e4)
        activation=sim.max(-1).values
        contribution=self.weights.softmax(-1)[None]*activation/0.2*CFG["prototype_k"]
        return dict(logits=contribution.sum(-1),contribution=contribution,maps=sim)

class PartModel(nn.Module):
    def __init__(self,variant):
        super().__init__(); self.variant=variant
        self.use_visibility=variant!="parts_independent"
        self.use_graph=variant in ["parts_graph","parts_graph_single", "parts_graph_detached", "parts_graph_single_detached"]
        self.detach_visibility=variant.endswith("_detached")
        self.hard_filter=False
        self.register_buffer("emission_threshold",torch.full((P,),CFG["visibility_threshold"]))
        self.register_buffer("anchors",anchors.clone())
        self.register_buffer("bank",part_bank.clone())
        self.register_buffer("support",part_support.clone())
        self.register_buffer("grid",GRID.clone())
        self.detector=nn.Sequential(nn.Linear(CFG["dim"],128),nn.GELU(),nn.Linear(128,P))
        nn.init.zeros_(self.detector[-1].weight); nn.init.zeros_(self.detector[-1].bias)
        self.part_embed=nn.Embedding(P,8)
        self.visibility_head=nn.Sequential(nn.Linear(CFG["dim"]+10,64),nn.GELU(),nn.Linear(64,1))
        self.class_weights=nn.Parameter(torch.zeros(C,P))
        self.relation_raw=nn.Parameter(torch.tensor(-1.5),requires_grad=self.use_graph)
        if self.use_graph:
            kernel=SINGLE_KERNELS if variant in ["parts_graph_single", "parts_graph_single_detached"] else GRAPH_KERNELS
            self.register_buffer("kernels",kernel.clone())
        self.register_buffer("src",torch.tensor([p for p,j in EDGES],dtype=torch.long))
        self.register_buffer("dst",torch.tensor([j for p,j in EDGES],dtype=torch.long))
        degree=torch.bincount(self.src,minlength=P).float().clamp_min(1)
        self.register_buffer("degree",degree)
    def set_policy(self,thresholds=None):
        self.hard_filter=thresholds is not None and self.use_visibility
        if thresholds is None:
            self.emission_threshold.fill_(CFG["visibility_threshold"])
        else:
            self.emission_threshold.copy_(torch.as_tensor(thresholds,device=self.emission_threshold.device,dtype=torch.float32))
        return self
    def visibility_logits(self,f,q,unary):
        d=torch.einsum("bpt,btd->bpd",q,f)
        peak=(q*unary).sum(-1,keepdim=True)
        entropy=-(q*q.clamp_min(1e-8).log()).sum(-1,keepdim=True)/math.log(T)
        emb=self.part_embed.weight[None].expand(len(f),-1,-1)
        return self.visibility_head(torch.cat([d,peak,entropy,emb],-1)).squeeze(-1)
    def forward(self,f,valid):
        f=F.normalize(f.float(),dim=-1)
        unary=(torch.einsum("btd,pd->bpt",f,self.anchors)+self.detector(f).transpose(1,2))/CFG["loc_temperature"]
        unary=unary.masked_fill(~valid[:,None,:],-1e4)
        q=unary.softmax(-1)
        if self.use_graph:
            neighbor_vis=self.visibility_logits(f,q,unary).sigmoid().detach()
            if getattr(self,"entropy_messages",False):
                entropy=-(q*q.clamp_min(1e-8).log()).sum(-1)/valid.sum(-1).clamp_min(2).log()[:,None]
                neighbor_vis=neighbor_vis*(1-entropy).clamp(0,1).detach()
            alpha=self.relation_raw.sigmoid()*CFG["relation_strength_max"]
            for _ in range(CFG["graph_steps"]):
                mass=torch.einsum("ets,bes->bet",self.kernels,q[:,self.dst,:])
                edge_message=mass.clamp_min(1e-6).log()*neighbor_vis[:,self.dst,None]
                messages=torch.zeros_like(q).index_add(1,self.src,edge_message)/self.degree[None,:,None]
                centered=(messages*valid[:,None,:]).sum(-1,keepdim=True)/valid.sum(-1)[:,None,None].clamp_min(1)
                messages=messages-centered
                q=(unary+alpha*messages).masked_fill(~valid[:,None,:],-1e4).softmax(-1)
        vis_logit=self.visibility_logits(f,q,unary)
        probability=vis_logit.sigmoid()
        gate=probability.detach() if self.detach_visibility else probability
        if not self.use_visibility: gate=torch.ones_like(probability)
        emitted=(gate>=self.emission_threshold[None]) if self.use_visibility else torch.ones_like(gate,dtype=torch.bool)
        if self.hard_filter:
            gate=gate*emitted.to(gate.dtype)
        # Raw policy retains all soft contributions; emitted is only the threshold diagnostic.
        # Hard policy actually zeros the rejected evidence in the classifier below.
        descriptor=F.normalize(torch.einsum("bpt,btd->bpd",q,f),dim=-1)
        sim=torch.einsum("bpd,cpmd->bcpm",descriptor,self.bank)
        sim=sim.masked_fill(~self.support[None],-1e4)
        best,mode=sim.max(-1)
        evidence=((best+1)/2).clamp(0,1)
        weights=self.class_weights.masked_fill(~self.support.any(-1),-1e4).softmax(-1)
        contribution=30.0*weights[None]*gate[:,None,:]*evidence
        xy=q@self.grid
        return dict(logits=contribution.sum(-1),contribution=contribution,q=q,xy=xy,
                    visibility_logits=vis_logit,gate=gate,emitted=emitted,mode=mode,evidence=evidence)

def make_model(variant): return FrozenPNP() if variant=="pnp_frozen" else PartModel(variant)

def objective(out,b):
    ce=F.cross_entropy(out["logits"],b["label"])
    if "q" not in out: return ce,dict(ce=float(ce.detach()))
    target=heatmap_targets(b["points"],b["valid"])
    per_part=-(target*out["q"].clamp_min(1e-8).log()).sum(-1)
    loc=(per_part*b["visible"]).sum()/b["visible"].sum().clamp_min(1)
    vis=F.binary_cross_entropy_with_logits(out["visibility_logits"],b["visible"])
    src=torch.tensor([a for a,b_ in UNDIRECTED],device=b["points"].device)
    dst=torch.tensor([b_ for a,b_ in UNDIRECTED],device=b["points"].device)
    good=b["visible"][:,src]*b["visible"][:,dst]
    predicted=out["xy"][:,dst]-out["xy"][:,src]
    truth=b["points"][:,dst]-b["points"][:,src]
    edge=(F.smooth_l1_loss(predicted,truth,reduction="none").sum(-1)*good).sum()/good.sum().clamp_min(1)
    total=ce+CFG["lambda_loc"]*loc+CFG["lambda_visibility"]*vis+CFG["lambda_edge"]*edge
    return total,dict(ce=float(ce.detach()),loc=float(loc.detach()),vis=float(vis.detach()),edge=float(edge.detach()))

# Suite models. Warm-up is ungated; the classifier is fitted after representations are frozen.
class MeanLinear(nn.Module):
    def __init__(self):
        super().__init__(); self.linear=nn.Linear(CFG["dim"],C)
    def forward(self,f,valid):
        pooled=(f*valid[:,:,None]).sum(1)/valid.sum(1)[:,None].clamp_min(1)
        logits=self.linear(F.normalize(pooled,dim=-1))
        return dict(logits=logits,contribution=logits[:,:,None])

class SuitePartModel(PartModel):
    def __init__(self,geometry,score):
        variant="parts_visibility_detached" if geometry=="none" else "parts_graph_detached"
        super().__init__(variant)
        self.geometry=geometry; self.score_mode=score
        self.use_visibility=False # base forward gets ungated evidence; scoring below owns the gate
        self.log_scale=nn.Parameter(torch.tensor(math.log(30.0)),requires_grad=False)
        self.entropy_messages=geometry=="entropy"
        if geometry=="single": self.kernels.copy_(SINGLE_KERNELS)
        elif geometry=="permuted":
            # Shuffle paired directional priors, preserve reverse pairing and node degree.
            count=len(UNDIRECTED); perm=np.random.default_rng(731).permutation(count)
            order=np.concatenate([perm,perm+count])
            self.kernels.copy_(GRAPH_KERNELS[torch.tensor(order)])
        elif geometry=="distance":
            delta=GRID[:,None,:]-GRID[None,:,:]
            kernel=torch.exp(-delta.square().sum(-1)/(2*.25**2))
            self.kernels.copy_(kernel[None].expand_as(self.kernels))
    def set_policy(self,thresholds=None):
        self.hard_filter=thresholds is not None and self.score_mode!="none"
        self.emission_threshold.copy_(torch.as_tensor(
            thresholds if thresholds is not None else np.full(P,.5),dtype=torch.float32,device=self.emission_threshold.device))
        return self
    def score_evidence(self,evidence,probability):
        gate=torch.ones_like(probability) if self.score_mode=="none" else probability.detach()
        emitted=torch.ones_like(gate,dtype=torch.bool) if self.score_mode=="none" else gate>=self.emission_threshold[None]
        if self.hard_filter: gate=gate*emitted
        weights=self.class_weights.masked_fill(~self.support.any(-1),-1e4).softmax(-1)
        weighted=weights[None]*gate[:,None,:]
        if self.score_mode=="normalized":
            weighted=weighted/weighted.sum(-1,keepdim=True).clamp_min(1e-6)
        scale=self.log_scale.clamp(math.log(1),math.log(100)).exp()
        contribution=scale*weighted*evidence
        return contribution,gate,emitted
    def forward(self,f,valid):
        out=super().forward(f,valid)
        contribution,gate,emitted=self.score_evidence(out["evidence"],out["visibility_logits"].sigmoid())
        out.update(logits=contribution.sum(-1),contribution=contribution,gate=gate,emitted=emitted)
        return out
    def freeze_representation(self):
        self.requires_grad_(False)
        self.class_weights.requires_grad_(True); self.log_scale.requires_grad_(True)
        return self

def make_suite(name):
    spec=SUITE[name]
    return MeanLinear() if spec["score"]=="linear" else SuitePartModel(spec["geometry"],spec["score"])


## Tests before spending GPU hours
Synthetic checks below verify: finite forward/backward for every model, normalized maps, padding exclusion, zero visible-keypoint loss handling, prototype banks as buffers, and exact additive logit decomposition. They do not measure accuracy. A separate annotation-alignment figure above checks the real input coordinate convention.

In [ ]:
# 9. Synthetic model sanity checks (no dataset labels are used in forward).
def run_sanity_checks():
    seed_all(777)
    for variant in EXPERIMENTS:
        model=make_model(variant).to(DEVICE)
        f=F.normalize(torch.randn(2,T,CFG["dim"],device=DEVICE),dim=-1)
        valid=torch.ones(2,T,dtype=torch.bool,device=DEVICE); valid[:,:N_GRID]=False
        b=dict(features=f,valid=valid,label=torch.tensor([0,min(1,C-1)],device=DEVICE),
               points=torch.rand(2,P,2,device=DEVICE),visible=torch.zeros(2,P,device=DEVICE))
        out=model(f,valid); loss,_=objective(out,b); loss.backward()
        assert torch.isfinite(loss) and torch.isfinite(out["logits"]).all()
        assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
        assert torch.allclose(out["logits"],out["contribution"].sum(-1),atol=1e-5)
        assert "bank" not in dict(model.named_parameters())
        if "q" in out:
            assert torch.allclose(out["q"].sum(-1),torch.ones(2,P,device=DEVICE),atol=1e-5)
            assert (out["q"][:,:,:N_GRID]==0).all()
        del model
    a=balanced_assign(torch.randn(101,5))
    assert torch.allclose(a.sum(-1),torch.ones(101),atol=1e-4)
    print("PASS: forward/backward, empty visibility, padding masks, fixed banks, additive explanations.")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
run_sanity_checks()

# Visibility-gradient test on synthetic data: check the intended learning route explicitly.
def test_visibility_gradients():
    for variant in ["parts_visibility","parts_visibility_detached","parts_graph_detached"]:
        model=make_model(variant).to(DEVICE)
        f=F.normalize(torch.randn(2,T,CFG["dim"],device=DEVICE),dim=-1)
        valid=torch.ones(2,T,dtype=torch.bool,device=DEVICE)
        out=model(f,valid)
        ce=F.cross_entropy(out["logits"],torch.tensor([0,min(1,C-1)],device=DEVICE))
        ce.backward()
        magnitude=sum(float(p.grad.abs().sum()) for p in model.visibility_head.parameters() if p.grad is not None)
        if variant.endswith("_detached"): assert magnitude==0, "Classification reached detached visibility head"
        else: assert magnitude>0, "Joint control lost its classification gradient"
        model.zero_grad(set_to_none=True)
        out=model(f,valid)
        F.binary_cross_entropy_with_logits(out["visibility_logits"],torch.zeros(2,P,device=DEVICE)).backward()
        assert any(p.grad is not None and p.grad.abs().sum()>0 for p in model.visibility_head.parameters())
        model.set_policy(np.full(P,1.01))
        with torch.no_grad(): hidden=model(f,valid)
        assert not hidden["emitted"].any()
        assert torch.count_nonzero(hidden["contribution"])==0
        assert torch.count_nonzero(hidden["logits"])==0
        del model
    print("PASS: joint/detached gradients, BCE gradients and actual hard rejection of all evidence.")
test_visibility_gradients()


for name in RUN_VARIANTS:
    model=make_suite(name).to(DEVICE)
    f=F.normalize(torch.randn(2,T,CFG["dim"],device=DEVICE),dim=-1)
    valid=torch.ones(2,T,dtype=torch.bool,device=DEVICE)
    out=model(f,valid); F.cross_entropy(out["logits"],torch.tensor([0,min(1,C-1)],device=DEVICE)).backward()
    assert torch.isfinite(out["logits"]).all()
    assert torch.allclose(out["logits"],out["contribution"].sum(-1),atol=1e-4)
    if hasattr(model,"freeze_representation"):
        model.freeze_representation()
        assert {k for k,p in model.named_parameters() if p.requires_grad}=={"class_weights","log_scale"}
        if model.score_mode!="none":
            model.set_policy(np.full(P,1.01))
            assert torch.count_nonzero(model(f,valid)["contribution"])==0
    del model
print("PASS: all suite forward/backward paths, exact contributions, phase-2 parameter isolation.")


## Data use and metrics

Selection rows choose checkpoints. Calibration rows choose per-part thresholds. Test rows (if enabled) generate the fixed experiment report. `fit_thresholds` asserts that only calibration IDs are used. Threshold fallback for scarce positive/negative examples is recorded.

Report accuracy, macro accuracy, PCK at the declared box-diagonal tolerance, missing emission rate, visible emission recall, localized emission recall, AUROC/AP/Brier, and abstention coverage. Keep raw and filtered policies distinct. The class-only linear model has no part metrics. Auxiliary visibility metrics of ungated models are diagnostic; their actual classifier emits all parts.

Epoch curves are for **selection**, including visibility Brier and PCK. A better Brier score with lower AUROC is a different tradeoff from better discrimination. Fixed-0.5 results and recall-controlled results answer different questions. PCK improvements are measured on the same held-out visible landmarks; geometry clusters are annotation-derived analysis slices only.


In [ ]:

def class_decision(out):
    pred=out["logits"].argmax(-1)
    # All-zero filtered support is explicit abstention, not class 0.
    if "gate" in out:
        pred=pred.masked_fill(out["gate"].sum(-1)<=1e-8,-1)
    return pred

# 10. Metrics and prediction collection.
@torch.no_grad()
def predict_rows(model,rows):
    model.eval(); records={k:[] for k in ["row","label","pred","logits"]}
    for raw in feature_loader(rows):
        b=move_batch(raw); out=model(b["features"],b["valid"])
        for k,value in [("row",b["row"]),("label",b["label"]),("pred",class_decision(out)),("logits",out["logits"])]:
            records[k].append(value.cpu().numpy())
        if "q" in out:
            xy=model.grid[out["q"].argmax(-1)]
            for k,value in [("xy",xy),("visibility",out["visibility_logits"].sigmoid()),("gate",out["gate"]),("emitted",out["emitted"])]:
                records.setdefault(k,[]).append(value.cpu().numpy())
    return {k:np.concatenate(v) for k,v in records.items()}

def compute_metrics(r):
    correct=(r["pred"]==r["label"])
    metrics=dict(accuracy=float(correct.mean()),macro_accuracy=float(np.mean([correct[r["label"]==c].mean() for c in np.unique(r["label"])])))
    if "xy" in r:
        rows=r["row"].astype(int); vis=visibility[rows]>0
        diag=np.sqrt((boxes[rows,2:]**2).sum(-1)).clip(1e-8)
        hit=np.linalg.norm(r["xy"]-points[rows],axis=-1)<=CFG["pck_fraction"]*diag[:,None]
        emitted=r.get("emitted",r["gate"]>=CFG["visibility_threshold"])
        def ratio(num,den): return float(num.sum()/den.sum()) if den.sum() else float("nan")
        metrics.update(pck=ratio(hit&vis,vis),missing_emission_rate=ratio(emitted&~vis,~vis),
                       visible_emission_recall=ratio(emitted&vis,vis),
                       localized_emission_recall=ratio(emitted&hit&vis,vis))
        metrics["visibility_auroc"]=float(roc_auc_score(vis.ravel(),r["visibility"].ravel())) if np.unique(vis).size==2 else float("nan")
    metrics["prediction_coverage"]=float((r["pred"]>=0).mean())
    accepted=r["pred"]>=0
    metrics["accepted_accuracy"]=float(correct[accepted].mean()) if accepted.any() else float("nan")
    if "visibility" in r:
        v=visibility[r["row"].astype(int)]>0
        score=r["visibility"]
        fixed=score>=0.5
        metrics.update(visibility_brier=float(np.mean((score-v.astype(float))**2)),
            visibility_ap=float(average_precision_score(v.ravel(),score.ravel())) if v.any() else float("nan"),
            visibility_visible_mean=float(score[v].mean()) if v.any() else float("nan"),
            visibility_invisible_mean=float(score[~v].mean()) if (~v).any() else float("nan"),
            visibility_tpr_at_05=float(fixed[v].mean()) if v.any() else float("nan"),
            visibility_fpr_at_05=float(fixed[~v].mean()) if (~v).any() else float("nan"))
    return metrics

def per_part_metrics(r):
    rows=r["row"].astype(int); vis=visibility[rows]>0
    diag=np.linalg.norm(boxes[rows,2:],axis=-1).clip(1e-8)
    hit=np.linalg.norm(r["xy"]-points[rows],axis=-1)<=CFG["pck_fraction"]*diag[:,None]
    emitted=r.get("emitted",r["gate"]>=CFG["visibility_threshold"])
    result=[]
    for p,name in enumerate(part_names):
        result.append(dict(part=name,n_visible=int(vis[:,p].sum()),n_invisible=int((~vis[:,p]).sum()),
            pck=float(hit[vis[:,p],p].mean()) if vis[:,p].any() else np.nan,
            missing_emission_rate=float(emitted[~vis[:,p],p].mean()) if (~vis[:,p]).any() else np.nan,
            visible_emission_recall=float(emitted[vis[:,p],p].mean()) if vis[:,p].any() else np.nan))
    return pd.DataFrame(result)

def choose_threshold(scores,visible,recall_target):
    """Use only validation observations. Highest threshold satisfying visible recall."""
    scores=np.asarray(scores,dtype=float); visible=np.asarray(visible,dtype=bool)
    if not visible.any(): return 0.5
    positives=np.sort(scores[visible])[::-1]
    required=max(1,int(np.ceil(recall_target*len(positives))))
    return float(positives[required-1])

def fit_thresholds(validation_record,recall_target=PRIMARY_RECALL):
    rows=validation_record["row"].astype(int)
    assert set(rows).issubset(set(calibration_idx)), "Threshold calibration attempted outside validation"
    scores=validation_record["visibility"]; vis=visibility[rows]>0
    pooled=choose_threshold(scores.ravel(),vis.ravel(),recall_target)
    thresholds=[]; report=[]
    for p,name in enumerate(part_names):
        nv=int(vis[:,p].sum()); ni=len(vis)-nv
        fallback=min(nv,ni)<CFG["threshold_min_examples"]
        t=pooled if fallback else choose_threshold(scores[:,p],vis[:,p],recall_target)
        thresholds.append(t); emit=scores[:,p]>=t
        report.append(dict(part=name,threshold=t,pooled_fallback=fallback,n_visible=nv,n_invisible=ni,
            validation_visible_recall=float(emit[vis[:,p]].mean()) if nv else None,
            validation_missing_emission=float(emit[~vis[:,p]].mean()) if ni else None))
    return np.array(thresholds,np.float32),report

def visibility_diagnostics(record,path):
    rows=record["row"].astype(int); truth=visibility[rows]>0; scores=record["visibility"]
    rows_out=[]
    for p,name in enumerate(part_names):
        y=truth[:,p]; s=scores[:,p]; pred=s>=.5
        rows_out.append(dict(part=name,visible_n=int(y.sum()),invisible_n=int((~y).sum()),
            tp=int((pred&y).sum()),fp=int((pred&~y).sum()),tn=int((~pred&~y).sum()),fn=int((~pred&y).sum()),
            brier=float(((s-y.astype(float))**2).mean()),
            auroc=float(roc_auc_score(y,s)) if len(np.unique(y))==2 else np.nan,
            visible_mean=float(s[y].mean()) if y.any() else np.nan,
            invisible_mean=float(s[~y].mean()) if (~y).any() else np.nan))
    pd.DataFrame(rows_out).to_csv(path,index=False)


In [ ]:

# 11. Shared, resumable representation learning + classifier refitting, then all policies.
PROTOCOL=dict(version="cub-suite-v3.1",cfg=CFG,suite=SUITE,run_variants=RUN_VARIANTS,
    head_epochs=HEAD_EPOCHS,linear_epochs=LINEAR_EPOCHS,recall_grid=CALIBRATION_RECALL_GRID,
    primary_recall=PRIMARY_RECALL,evaluate_test=EVALUATE_TEST,splits=split_manifest,cache=CACHE_KEY)
RUN_KEY=stable_hash(PROTOCOL)
RUNS=OUT/"runs"; WARM=OUT/"representations"
RUNS.mkdir(exist_ok=True); WARM.mkdir(exist_ok=True)
save_json(PROTOCOL,OUT/"locked_protocol.json")
if "posthoc_soft" in RUN_VARIANTS:
    assert "independent" in RUN_VARIANTS and RUN_VARIANTS.index("independent")<RUN_VARIANTS.index("posthoc_soft")

def capture_rng(generator):
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,loader=generator.get_state())
def restore_rng(r,generator):
    random.setstate(r["python"]); np.random.set_state(r["numpy"]); torch.set_rng_state(r["torch"])
    if torch.cuda.is_available() and r["cuda"] is not None: torch.cuda.set_rng_state_all(r["cuda"])
    generator.set_state(r["loader"])

def fit_phase(model,directory,epochs,phase,seed):
    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)
    generator=torch.Generator().manual_seed(seed)
    lr=.01 if phase=="linear" else CFG["lr"]
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=CFG["weight_decay"])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    epoch0=0;best=-np.inf;history=[];elapsed_total=0.
    if RESUME_INPUT and AUTO_RESUME:
        relative=directory.relative_to(OUT)
        source=Path(RESUME_INPUT)/relative
        for name in ["last.pt","best.pt","history.csv"]:
            if not (directory/name).exists() and (source/name).exists(): shutil.copy2(source/name,directory/name)
    if AUTO_RESUME and (directory/"last.pt").exists():
        ck=load_local_checkpoint(directory/"last.pt")
        assert ck["run_key"]==RUN_KEY and ck["phase"]==phase,"Incompatible checkpoint: use a new OUT or the original configuration"
        model.load_state_dict(ck["model"]);opt.load_state_dict(ck["optimizer"]);sched.load_state_dict(ck["scheduler"])
        for state in opt.state.values():
            for key,value in state.items():
                if torch.is_tensor(value) and key!="step":state[key]=value.to(DEVICE)
        epoch0=ck["epoch"]+1;best=ck["best"];history=ck["history"];elapsed_total=ck["seconds"]
        restore_rng(ck["rng"],generator)
    loader=feature_loader(train_idx,shuffle=True,generator=generator)
    for epoch in range(epoch0,epochs):
        model.train();start=time.time();total=0.;count=0;terms={}
        if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
        for raw in tqdm(loader,desc=f"{directory.name}/{phase} {epoch+1}/{epochs}",leave=False):
            b=move_batch(raw);opt.zero_grad(set_to_none=True);out=model(b["features"],b["valid"])
            if phase=="representation":loss,components=objective(out,b)
            else:
                loss=F.cross_entropy(out["logits"],b["label"]);components={"ce":float(loss.detach())}
            if not torch.isfinite(loss):raise FloatingPointError(f"Nonfinite {phase} loss")
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.0);opt.step()
            bs=len(b["label"]);total+=float(loss.detach())*bs;count+=bs
            for k,v in components.items():terms[k]=terms.get(k,0.)+v*bs
        sched.step();selection=compute_metrics(predict_rows(model,selection_idx));score=selection["accuracy"]
        seconds=time.time()-start;elapsed_total+=seconds
        row=dict(epoch=epoch+1,loss=total/count,seconds=seconds,
            **{f"train_{k}":v/count for k,v in terms.items()},**{f"selection_{k}":v for k,v in selection.items()},
            peak_gpu_gib=torch.cuda.max_memory_allocated()/2**30 if torch.cuda.is_available() else 0.)
        history.append(row);pd.DataFrame(history).to_csv(directory/"history.csv",index=False)
        if score>best:
            best=score;atomic_torch_save(dict(run_key=RUN_KEY,model=model.state_dict(),epoch=epoch+1,
                phase=phase,selection=selection),directory/"best.pt")
        atomic_torch_save(dict(run_key=RUN_KEY,model=model.state_dict(),optimizer=opt.state_dict(),
            scheduler=sched.state_dict(),epoch=epoch,best=best,history=history,seconds=elapsed_total,
            phase=phase,rng=capture_rng(generator)),directory/"last.pt")
        print(f"{directory.name} {phase} {epoch+1}: selection acc={score:.4f}"+
              (f", PCK={selection['pck']:.4f}, Brier={selection['visibility_brier']:.4f}" if 'pck' in selection else ""))
    best_ck=load_local_checkpoint(directory/"best.pt");model.load_state_dict(best_ck["model"])
    return dict(best_epoch=best_ck["epoch"],seconds=elapsed_total,selection=best_ck["selection"])

def warm_representation(geometry,seed):
    seed_all(seed);model=SuitePartModel(geometry,"none").to(DEVICE)
    directory=WARM/f"{geometry}_seed{seed}"
    info=fit_phase(model,directory,CFG["epochs"],"representation",seed)
    return model,info

def frozen_fingerprint(model):
    h=hashlib.sha256()
    for k,p in model.named_parameters():
        if k not in ["class_weights","log_scale"]:h.update(p.detach().cpu().numpy().tobytes())
    return h.hexdigest()

def train_suite(name,seed):
    seed_all(seed);spec=SUITE[name];directory=RUNS/f"{name}_seed{seed}";directory.mkdir(exist_ok=True)
    if spec["score"]=="linear":
        model=MeanLinear().to(DEVICE);info=fit_phase(model,directory,LINEAR_EPOCHS,"linear",seed);warm_info={}
    elif spec.get("posthoc"):
        model=make_suite(name).to(DEVICE)
        ck=load_local_checkpoint(RUNS/f"independent_seed{seed}"/"best.pt")
        assert ck["run_key"]==RUN_KEY
        model.load_state_dict(ck["model"]);model.score_mode="soft"
        info=dict(best_epoch=ck["epoch"],seconds=0.,selection=compute_metrics(predict_rows(model,selection_idx)))
        warm_info={};atomic_torch_save(dict(ck,model=model.state_dict(),phase="posthoc"),directory/"best.pt")
    else:
        model,warm_info=warm_representation(spec["geometry"],seed)
        model.score_mode=spec["score"];model.freeze_representation();before=frozen_fingerprint(model)
        seed_all(seed+10000)
        info=fit_phase(model,directory,HEAD_EPOCHS,"head",seed+10000)
        assert before==frozen_fingerprint(model),"A supposedly frozen representation changed during head training"
    report=predict_rows(model,REPORT_ROWS)
    np.savez_compressed(directory/"report_raw.npz",**report)
    common=dict(variant=name,seed=seed,split=REPORT_SPLIT,best_epoch=info["best_epoch"],
        phase2_seconds=info["seconds"],shared_representation_seconds=warm_info.get("seconds",0.),
        parameter_count=sum(p.numel() for p in model.parameters()))
    records=[dict(common,policy="raw",recall_target=np.nan,**compute_metrics(report))]
    if "visibility" in report:
        calibration=predict_rows(model,calibration_idx)
        np.savez_compressed(directory/"calibration_raw.npz",**calibration)
        visibility_diagnostics(calibration,directory/"calibration_visibility.csv")
        visibility_diagnostics(report,directory/"report_visibility.csv")
        per_part_metrics(report).to_csv(directory/"parts_raw.csv",index=False)
        policies={}
        for recall in CALIBRATION_RECALL_GRID:
            thresholds,details=fit_thresholds(calibration,recall)
            policy=f"filtered_{recall:.2f}";policies[policy]=dict(thresholds=thresholds.tolist(),details=details,recall=recall)
            model.set_policy(thresholds)
            filtered=predict_rows(model,REPORT_ROWS)
            np.savez_compressed(directory/f"report_{policy}.npz",**filtered)
            records.append(dict(common,policy=policy,recall_target=recall,**compute_metrics(filtered)))
            if recall==PRIMARY_RECALL:per_part_metrics(filtered).to_csv(directory/"parts_primary.csv",index=False)
        save_json(dict(source="calibration_only",policies=policies),directory/"thresholds.json")
        # Oracle visibility is DIAGNOSTIC ONLY on calibration; no GT gates are deployed.
        model.set_policy(None);correct=0;total=0
        with torch.no_grad():
            for raw in feature_loader(calibration_idx):
                b=move_batch(raw);out=model(b["features"],b["valid"])
                contribution,gate,emit=model.score_evidence(out["evidence"],b["visible"])
                pred=class_decision(dict(logits=contribution.sum(-1),gate=gate))
                correct+=int((pred==b["label"]).sum());total+=len(pred)
        save_json(dict(calibration_oracle_gate_accuracy=correct/total,n=total,
            caveat="Posthoc GT gate diagnostic, not an upper bound or deployable model; ungated models ignore the gate"),directory/"oracle_gate_diagnostic.json")
    save_json(dict(common,phase_info=info,warm_info=warm_info),directory/"training_info.json")
    pd.DataFrame(records).to_csv(directory/"metrics.csv",index=False)
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return records

results=[]
for seed in CFG["seeds"]:
    for name in RUN_VARIANTS:
        results.extend(train_suite(name,seed))
        pd.DataFrame(results).to_csv(OUT/"all_results.csv",index=False)
results=pd.DataFrame(results)
primary=results[results.policy.isin(["raw",f"filtered_{PRIMARY_RECALL:.2f}"])]
display(primary[["variant","policy","seed","accuracy","pck","missing_emission_rate","visible_emission_recall","visibility_brier"]])
print("All configured runs finished. No variant or operating point is selected using test scores.")


## Graphical and statistical analysis

Plots: accuracy and PCK with seed variation; accuracy versus invisible-part emission tradeoffs; visibility reliability diagrams; per-part PCK heatmaps; selection trajectories. Raw and filtered rows are separate. The fixed primary graph comparison is `graph_normalized` minus `staged_normalized`. Wrong-prior and distance controls test whether anatomical edge information adds value.

Paired bootstrap resamples images, retaining part correlation within each image. These intervals are conditional on each trained seed; report seed variation too. Multiple exploratory contrasts are not independent confirmations. No p-value or acceptance claim is manufactured. Geometry-proxy slices below are annotation-based 2D layout groups, not ground-truth flight or viewpoint labels.


In [ ]:

# 12. Fixed comparisons, plots, and train-fitted 2D layout slices.
def read_report(name,seed,policy="raw"):
    with np.load(RUNS/f"{name}_seed{seed}"/f"report_{policy}.npz") as data:return {k:data[k] for k in data.files}

def paired_intervals(a,b,n_boot=1000):
    assert np.array_equal(a["row"],b["row"])
    rng=np.random.default_rng(999);n=len(a["row"])
    acc=(a["pred"]==a["label"]).astype(float)-(b["pred"]==b["label"]).astype(float)
    rows=a["row"];v=visibility[rows]>0;den=v.sum(-1);diag=np.linalg.norm(boxes[rows,2:],axis=-1).clip(1e-8)
    hits=[]
    for record in [a,b]:
        hit=np.linalg.norm(record["xy"]-points[rows],axis=-1)<=CFG["pck_fraction"]*diag[:,None]
        hits.append((hit&v).sum(-1))
    delta=hits[0]-hits[1];samples=[]
    for _ in range(n_boot):
        idx=rng.integers(n,size=n)
        samples.append([acc[idx].mean(),delta[idx].sum()/max(1,den[idx].sum())])
    samples=np.asarray(samples)
    return dict(accuracy_delta=float(acc.mean()),accuracy_low=float(np.quantile(samples[:,0],.025)),accuracy_high=float(np.quantile(samples[:,0],.975)),
        pck_delta=float(delta.sum()/max(1,den.sum())),pck_low=float(np.quantile(samples[:,1],.025)),pck_high=float(np.quantile(samples[:,1],.975)))

comparisons=[("staged_soft","posthoc_soft"),("staged_normalized","staged_soft"),
    ("graph_ungated","independent"),("graph_normalized","staged_normalized"),
    ("graph_normalized","single_graph_normalized"),("graph_normalized","permuted_graph_normalized"),
    ("graph_normalized","distance_graph_normalized"),("entropy_graph_normalized","graph_normalized")]
intervals=[]
for a,b in comparisons:
    if not {a,b}.issubset(RUN_VARIANTS):continue
    for seed in CFG["seeds"]:
        for policy in ["raw",f"filtered_{PRIMARY_RECALL:.2f}"]:
            intervals.append(dict(a=a,b=b,seed=seed,policy=policy,**paired_intervals(read_report(a,seed,policy),read_report(b,seed,policy))))
pd.DataFrame(intervals).to_csv(OUT/"paired_intervals.csv",index=False)
summary=results.groupby(["variant","policy"])[["accuracy","pck","missing_emission_rate","visible_emission_recall","visibility_brier","prediction_coverage"]].agg(["mean","std"])
summary.to_csv(OUT/"summary.csv");display(summary)

fig,axes=plt.subplots(1,2,figsize=(14,5))
raw=results[results.policy.eq("raw")]
for ax,metric in zip(axes,["accuracy","pck"]):
    grouped=raw.groupby("variant")[metric].agg(["mean","std"]).dropna(subset=["mean"])
    ax.barh(grouped.index,100*grouped["mean"],xerr=100*grouped["std"].fillna(0))
    ax.set_xlabel(metric+" (%) — seed SD");ax.grid(axis="x",alpha=.2)
plt.tight_layout();plt.savefig(OUT/"accuracy_localization.png",dpi=160);plt.show()

fig,axes=plt.subplots(1,2,figsize=(13,5))
for name in RUN_VARIANTS:
    if name=="mean_linear":continue
    g=results[(results.variant==name)&results.policy.ne("raw")].groupby("recall_target").mean(numeric_only=True)
    axes[0].plot(100*g.missing_emission_rate,100*g.accuracy,"o-",label=name)
    axes[1].plot(100*g.visible_emission_recall,100*g.missing_emission_rate,"o-",label=name)
axes[0].set(xlabel="Invisible-part emission (%) ↓",ylabel="Classification accuracy (%) ↑")
axes[1].set(xlabel="Visible-part emission recall (%) ↑",ylabel="Invisible-part emission (%) ↓")
axes[0].legend(fontsize=7);axes[1].grid(alpha=.2)
plt.tight_layout();plt.savefig(OUT/"tradeoff_curves.png",dpi=160);plt.show()

part_matrix={};reliability=[]
fig,ax=plt.subplots(figsize=(7,5));ax.plot([0,1],[0,1],"k--",label="ideal")
for name in RUN_VARIANTS:
    if name=="mean_linear":continue
    r=read_report(name,CFG["seeds"][0]);y=(visibility[r["row"]]>0).ravel();score=r["visibility"].ravel()
    xx=[];yy=[]
    for lo in np.linspace(0,.9,10):
        mask=(score>=lo)&(score<(lo+.1) if lo<.89 else score<=1)
        if mask.any():
            x=float(score[mask].mean());z=float(y[mask].mean());xx.append(x);yy.append(z)
            reliability.append(dict(variant=name,bin_left=float(lo),n=int(mask.sum()),confidence=x,observed_visibility=z))
    ax.plot(xx,yy,"o-",label=name,alpha=.75)
    part_matrix[name]=per_part_metrics(r).set_index("part").pck
ax.set(xlabel="Predicted visibility",ylabel="Observed visible fraction");ax.legend(fontsize=6)
plt.tight_layout();plt.savefig(OUT/"visibility_reliability.png",dpi=160);plt.show()
pd.DataFrame(reliability).to_csv(OUT/"reliability_bins.csv",index=False)
part_table=pd.DataFrame(part_matrix);part_table.to_csv(OUT/"per_part_pck_matrix.csv")
fig,ax=plt.subplots(figsize=(12,6));im=ax.imshow(part_table.values,vmin=0,vmax=1,cmap="viridis",aspect="auto")
ax.set_yticks(range(P),part_names);ax.set_xticks(range(len(part_table.columns)),part_table.columns,rotation=70,ha="right")
fig.colorbar(im,ax=ax,label="PCK");plt.tight_layout();plt.savefig(OUT/"part_localization_heatmap.png",dpi=160);plt.show()

fig,axes=plt.subplots(1,3,figsize=(15,4))
for geometry in ["none","anatomy"]:
    for seed in CFG["seeds"]:
        path=OUT/"representations"/f"{geometry}_seed{seed}"/"history.csv"
        if not path.exists():continue
        hist=pd.read_csv(path)
        for ax,metric in zip(axes,["accuracy","pck","visibility_brier"]):
            ax.plot(hist.epoch,hist[f"selection_{metric}"],label=f"{geometry}/{seed}")
            ax.set(xlabel="Representation epoch",ylabel="Selection "+metric)
axes[0].legend(fontsize=7);plt.tight_layout()
plt.savefig(OUT/"selection_trajectories.png",dpi=160);plt.show()

# Same source token and directed edge across controls; these are learned training priors.
fig,axes=plt.subplots(1,4,figsize=(14,3))
side=int(round(T**.5));center=(side//2)*side+side//2
for ax,name in zip(axes,["graph_normalized","single_graph_normalized","permuted_graph_normalized","distance_graph_normalized"]):
    model=make_suite(name)
    ax.imshow(model.kernels[0,center].detach().cpu().numpy().reshape(side,side),cmap="magma")
    ax.set_title(name.replace("_normalized",""),fontsize=8);ax.axis("off")
plt.tight_layout();plt.savefig(OUT/"geometry_prior_controls.png",dpi=160);plt.show()

# Fit geometry proxies from training annotations only; these are evaluation slices, never model inputs.
src=np.array([a for a,b in UNDIRECTED]);dst=np.array([b for a,b in UNDIRECTED])
v=(visibility[:,src]>0)&(visibility[:,dst]>0)
d=(points[:,dst]-points[:,src])/np.linalg.norm(boxes[:,2:],axis=-1).clip(1e-6)[:,None,None]
x=np.where(v[:,:,None],d,np.nan).reshape(len(frame),-1)
mean=np.nanmean(x[train_idx],axis=0);mean=np.nan_to_num(mean)
x=np.where(np.isfinite(x),x,mean);scale=x[train_idx].std(0).clip(.05)
x=(x-mean)/scale
km=KMeans(n_clusters=min(6,len(train_idx)),n_init=10,random_state=211).fit(x[train_idx])
layout=km.predict(x);distance=np.linalg.norm(x-km.cluster_centers_[layout],axis=-1)
cutoff=float(np.quantile(distance[train_idx],.9))
save_json(dict(means=mean.tolist(),scales=scale.tolist(),centers=km.cluster_centers_.tolist(),distance90=cutoff,
    caveat="2D layout/missingness proxies; not ground-truth view or flight labels"),OUT/"layout_slice_definition.json")
slices=[]
for name in RUN_VARIANTS:
    for seed in CFG["seeds"]:
        r=read_report(name,seed);rows=r["row"]
        groups=[(f"layout_{k}",layout[rows]==k) for k in range(km.n_clusters)]
        groups += [("high_layout_distance",distance[rows]>cutoff),("low_visibility",visibility[rows].sum(-1)<=7)]
        for label,mask in groups:
            if mask.any():slices.append(dict(variant=name,seed=seed,slice=label,n=int(mask.sum()),**compute_metrics({k:v[mask] for k,v in r.items()})))
pd.DataFrame(slices).to_csv(OUT/"geometry_visibility_slices.csv",index=False)


## Qualitative review
Gallery images are chosen deterministically from the reporting split, not by correctness. The graph model's primary filtered policy is shown. Part contributions sum to the class logit; margin contributions sum to predicted-minus-runner-up logit. Hidden parts have zero explicit contribution. These properties do not establish pixel-level causal faithfulness. Ground-truth labels in plot titles are for evaluation only.

In [ ]:
# 13. Inference helpers and explanation gallery.
def load_best(variant,seed,policy="raw"):
    model=make_suite(variant).to(DEVICE)
    ck=load_local_checkpoint(RUNS/f"{variant}_seed{seed}"/"best.pt")
    assert ck["run_key"]==RUN_KEY
    model.load_state_dict(ck["model"])
    if policy!="raw" and hasattr(model,"set_policy"):
        thresholds=json.loads((RUNS/f"{variant}_seed{seed}"/"thresholds.json").read_text())["policies"][policy]["thresholds"]
        model.set_policy(thresholds)
    return model.eval()

@torch.no_grad()
def explain_one(model,index,save=True):
    b=move_batch(next(iter(feature_loader([index]))))
    out=model(b["features"],b["valid"])
    pred=int(class_decision(out)[0]); truth=int(frame.iloc[index].label)
    if pred<0:
        print(f"Image {int(frame.iloc[index].id)}: ABSTAIN — all part evidence rejected.")
        im,_=letterbox(index,True); display(im); return None
    runner=int(out["logits"][0].topk(2).indices[1])
    im,_=letterbox(index,True)
    xy=model.grid[out["q"].argmax(-1)][0].cpu().numpy()*CFG["image_size"]
    gate=out["gate"][0].cpu().numpy(); contribution=out["contribution"][0,pred].cpu().numpy()
    fig,axes=plt.subplots(1,4,figsize=(21,4.5)); axes[0].imshow(im); axes[1].imshow(im)
    emitted=out["emitted"][0].cpu().numpy()
    for p in range(P):
        if emitted[p]:
            axes[0].scatter(*xy[p],s=18,c="yellow"); axes[0].text(*xy[p],str(p+1),color="black",fontsize=8,
                bbox=dict(facecolor="white",alpha=.7,pad=.3))
    for p,j in UNDIRECTED:
        if emitted[p] and emitted[j]:
            axes[0].plot(xy[[p,j],0],xy[[p,j],1],color="cyan",alpha=.5,lw=1)
    top=int(np.argmax(contribution))
    heat=out["q"][0,top].reshape(N_GRID,N_GRID).cpu().numpy()
    axes[1].imshow(heat,extent=(0,CFG["image_size"],CFG["image_size"],0),cmap="magma",alpha=.5)
    axes[1].set_title(f"Top contribution: {part_names[top]} (gate={gate[top]:.2f})")
    order=np.argsort(contribution); axes[2].barh(np.array(part_names)[order],contribution[order]); axes[2].set_xlabel("Contribution to predicted-class logit")
    margin=(out["contribution"][0,pred]-out["contribution"][0,runner]).cpu().numpy()
    ranking=np.argsort(margin)
    axes[3].barh(np.array(part_names)[ranking],margin[ranking],color=np.where(margin[ranking]>=0,"teal","salmon"))
    axes[3].axvline(0,color="black",lw=.7)
    axes[3].set_title(f"Predicted minus {class_names[runner]}",fontsize=8)
    axes[3].set_xlabel("Contribution to class logit margin")
    assert np.isclose(margin.sum(),float(out["logits"][0,pred]-out["logits"][0,runner]),atol=1e-4)
    axes[0].set_title(f"Pred: {class_names[pred]}\nTrue: {class_names[truth]}",fontsize=9)
    for ax in axes[:2]: ax.axis("off")
    assert np.isclose(contribution.sum(),float(out["logits"][0,pred]),atol=1e-4)
    plt.tight_layout()
    if save: plt.savefig(OUT/f"explanation_id{int(frame.iloc[index].id)}.png",dpi=160,bbox_inches="tight")
    plt.show()
    return pred,top,int(out["mode"][0,pred,top])

@torch.no_grad()
def retrieve_exemplar(c,p,mode):
    proto=part_bank[c,p,mode].numpy(); best=(-np.inf,None,None)
    for i in train_idx[(frame.label.to_numpy()[train_idx]==c)&(visibility[train_idx,p]>0)]:
        f=np.asarray(features[i],dtype=np.float32)
        # Restrict to a small neighborhood of this TRAINING landmark.
        distance=np.linalg.norm(GRID.numpy()-points[i,p],axis=-1)
        allowed=valid_tokens[i]&(distance<=1.5/N_GRID)
        if not allowed.any(): allowed[np.argmin(distance)]=True
        sim=f@proto; sim[~allowed]=-np.inf; t=int(sim.argmax())
        if sim[t]>best[0]: best=(float(sim[t]),int(i),t)
    if best[1] is None: return None
    score,i,t=best; im,_=letterbox(i,True); x,y=GRID[t].numpy()*CFG["image_size"]
    radius=CFG["image_size"]*.12
    crop=im.crop((max(0,int(x-radius)),max(0,int(y-radius)),min(im.width,int(x+radius)),min(im.height,int(y+radius))))
    return crop,dict(training_id=int(frame.iloc[i].id),part=part_names[p],similarity=score)

if "graph_normalized" in RUN_VARIANTS:
    viz_model=load_best("graph_normalized",CFG["seeds"][0],policy=f"filtered_{PRIMARY_RECALL:.2f}")
    gallery=np.random.default_rng(765).choice(REPORT_ROWS,min(3,len(REPORT_ROWS)),replace=False)
    for i in gallery:
        explanation=explain_one(viz_model,int(i))
        if explanation is None: continue
        pred,p,mode=explanation
        exemplar=retrieve_exemplar(pred,p,mode)
        if exemplar is not None:
            crop,info=exemplar; display(crop); print("Training exemplar:",info)
    del viz_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Robustness batch: three occlusion sizes and blur
Use the same image subset and same targeted landmark for all models, seeds, and severities. Test 8%, 14%, and 20% image-side squares, each with an equal-area random-location control, plus global Gaussian blur. Squares can cover multiple landmarks; record the actual count. These are diagnostics on pixel corruptions, not proof of genuine unseen-view robustness or semantic counterfactuals. Thresholds remain frozen from clean calibration.

Report clean and corrupted accuracy, conditional accuracy on each model's clean-correct examples, missing/covered emission, surviving visible recall, and abstention. This prevents a weaker clean model from appearing robust just because it already makes more errors. No corruption is used for training in this suite.


In [ ]:

# 14. Pixel-space corruptions, cached once, shared across all seeds/models.
from PIL import ImageFilter
class CorruptionRows(Dataset):
    def __init__(self,rows,centers,side,condition):
        self.rows=np.asarray(rows);self.centers=centers;self.side=side;self.condition=condition
    def __len__(self):return len(self.rows)
    def __getitem__(self,j):
        i=int(self.rows[j]);im,_=letterbox(i,True);size=CFG["image_size"]
        rect=np.array([-1,-1,-1,-1],np.float32)
        if self.condition=="blur":im=im.filter(ImageFilter.GaussianBlur(radius=2.0))
        else:
            width=max(2,round(size*self.side));cx,cy=self.centers[j]*size
            left=int(np.clip(round(cx-width/2),0,size-width));top=int(np.clip(round(cy-width/2),0,size-width))
            draw=ImageDraw.Draw(im);draw.rectangle([left,top,left+width-1,top+width-1],fill=tuple(int(v*255) for v in MEAN))
            rect=np.array([left,top,left+width,top+width],np.float32)/size
        x=torch.from_numpy(((np.asarray(im).astype(np.float32)/255-MEAN)/STD).transpose(2,0,1).copy())
        return x,j,torch.tensor(rect)

@torch.no_grad()
def corruption_cache(backbone,rows,centers,side,condition,path):
    if path.exists():
        with np.load(path) as data:return data["features"],data["rects"]
    output=np.zeros((len(rows),T,CFG["dim"]),np.float16);rects=np.zeros((len(rows),4),np.float32)
    loader=DataLoader(CorruptionRows(rows,centers,side,condition),batch_size=CFG["extraction_batch"],num_workers=CFG["workers"])
    for image,j,rect in tqdm(loader,desc=path.stem):
        with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=AMP_ENABLED):
            f=backbone.forward_features(image.to(DEVICE))["x_norm_patchtokens"]
        output[j.numpy()]=F.normalize(f.float(),dim=-1).cpu().numpy().astype(np.float16);rects[j.numpy()]=rect.numpy()
    tmp=path.with_suffix(".tmp.npz");np.savez_compressed(tmp,features=output,rects=rects);tmp.replace(path)
    return output,rects

@torch.no_grad()
def array_prediction(model,array,rows):
    preds=[];emits=[];gates=[]
    for start in range(0,len(rows),CFG["batch_size"]):
        rr=rows[start:start+CFG["batch_size"]]
        f=torch.tensor(np.asarray(array[start:start+CFG["batch_size"]],dtype=np.float32),device=DEVICE)
        out=model(f,torch.tensor(valid_tokens[rr],device=DEVICE));preds.append(class_decision(out).cpu().numpy())
        if "gate" in out:gates.append(out["gate"].cpu().numpy());emits.append(out["emitted"].cpu().numpy())
    return np.concatenate(preds),np.concatenate(gates) if gates else None,np.concatenate(emits) if emits else None

if RUN_ROBUSTNESS:
    rng=np.random.default_rng(859);candidates=REPORT_ROWS[visibility[REPORT_ROWS].sum(-1)>0]
    robust_rows=np.sort(rng.choice(candidates,min(CFG["robustness_n"],len(candidates)),replace=False))
    target_parts=np.array([rng.choice(np.flatnonzero(visibility[i])) for i in robust_rows])
    centers_target=points[robust_rows,target_parts]
    centers_random=np.stack([GRID.numpy()[rng.choice(np.flatnonzero(valid_tokens[i]))] for i in robust_rows])
    corrupt_dir=OUT/"corruption_cache";corrupt_dir.mkdir(exist_ok=True)
    conditions=[(kind,side,centers_target if kind=="target" else centers_random) for side in OCCLUSION_SIDES for kind in ["target","random"]]
    conditions.append(("blur",0.,centers_random))
    cached={};backbone=None
    for kind,side,centers in conditions:
        key=stable_hash(dict(cache=CACHE_KEY,rows=robust_rows.tolist(),centers=centers.tolist(),side=side,kind=kind,version="pixel-v3"))
        path=corrupt_dir/f"{kind}_{side}_{key}.npz"
        if not path.exists() and backbone is None:backbone=load_backbone()
        cached[(kind,side)]=corruption_cache(backbone,robust_rows,centers,side,kind,path)
    del backbone;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    save_json(dict(image_ids=frame.iloc[robust_rows].id.tolist(),target_part_ids=(target_parts+1).tolist(),
        sides=OCCLUSION_SIDES,split=REPORT_SPLIT),OUT/"robustness_protocol.json")
    robust_metrics=[];per_image=[]
    truth=frame.label.to_numpy()[robust_rows]
    for seed in CFG["seeds"]:
        for name in RUN_VARIANTS:
            for policy in (["raw"] if name=="mean_linear" else ["raw",f"filtered_{PRIMARY_RECALL:.2f}"]):
                model=load_best(name,seed,policy)
                clean,clean_gate,clean_emit=array_prediction(model,features[robust_rows],robust_rows)
                clean_correct=clean==truth
                for (kind,side),(f,rects) in cached.items():
                    pred,gate,emit=array_prediction(model,f,robust_rows)
                    hit=((points[robust_rows,:,0]>=rects[:,None,0])&(points[robust_rows,:,0]<rects[:,None,2])&
                        (points[robust_rows,:,1]>=rects[:,None,1])&(points[robust_rows,:,1]<rects[:,None,3])&(visibility[robust_rows]>0))
                    surviving=(visibility[robust_rows]>0)&~hit
                    record=dict(variant=name,seed=seed,policy=policy,condition=kind,side=side,n=len(robust_rows),
                        clean_accuracy=float(clean_correct.mean()),corrupted_accuracy=float((pred==truth).mean()),
                        clean_correct_n=int(clean_correct.sum()),accuracy_on_clean_correct=float((pred[clean_correct]==truth[clean_correct]).mean()) if clean_correct.any() else np.nan,
                        prediction_flip=float((pred!=clean).mean()),coverage=float((pred>=0).mean()),mean_landmarks_covered=float(hit.sum(-1).mean()))
                    if emit is not None:
                        record.update(covered_emission=float(emit[hit].mean()) if hit.any() else np.nan,
                            surviving_visible_recall=float(emit[surviving].mean()) if surviving.any() else np.nan,
                            target_gate_drop=float((clean_gate[np.arange(len(robust_rows)),target_parts]-gate[np.arange(len(robust_rows)),target_parts]).mean()))
                    robust_metrics.append(record)
                    per_image.extend(dict(variant=name,seed=seed,policy=policy,condition=kind,side=side,
                        image_id=int(frame.iloc[i].id),label=int(y),clean=int(cp),corrupted=int(pp),landmarks_covered=int(h))
                        for i,y,cp,pp,h in zip(robust_rows,truth,clean,pred,hit.sum(-1)))
                del model
    robust_metrics=pd.DataFrame(robust_metrics);robust_metrics.to_csv(OUT/"robustness_summary.csv",index=False)
    pd.DataFrame(per_image).to_csv(OUT/"robustness_per_image.csv",index=False)
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    for name in RUN_VARIANTS:
        d=robust_metrics[(robust_metrics.variant==name)&(robust_metrics.condition=="target")&(robust_metrics.policy=="raw")].groupby("side").mean(numeric_only=True)
        axes[0].plot(d.index,100*d.corrupted_accuracy,"o-",label=name)
        if "covered_emission" in d:axes[1].plot(d.index,100*d.covered_emission,"o-",label=name)
    axes[0].set(xlabel="Target square side / image side",ylabel="Corrupted accuracy (%)")
    axes[1].set(xlabel="Target square side / image side",ylabel="Covered-part emission (%)")
    axes[0].legend(fontsize=6);plt.tight_layout();plt.savefig(OUT/"occlusion_severity_curves.png",dpi=160);plt.show()
    fig,axes=plt.subplots(1,4,figsize=(12,3))
    im,_=letterbox(int(robust_rows[0]),True);axes[0].imshow(im);axes[0].set_title("Clean")
    for ax,side in zip(axes[1:],OCCLUSION_SIDES):
        image,_,_=CorruptionRows(robust_rows,centers_target,side,"target")[0]
        ax.imshow(np.clip(image.numpy().transpose(1,2,0)*STD+MEAN,0,1));ax.set_title(f"Target {side:.0%}")
    for ax in axes:ax.axis("off")
    plt.tight_layout();plt.savefig(OUT/"occlusion_examples.png",dpi=160);plt.show()


## What decisions this batch supports

1. **Staging:** staged-soft versus posthoc-soft with the same ungated representation. If staging restores accuracy, distribution mismatch is a useful explanation; it does not alone establish novelty.
2. **Normalization:** normalized versus unnormalized staged scores. Keep only if the accuracy–rejection frontier improves without relying on a narrow threshold.
3. **Anatomy:** graph versus no graph, single-mode, mismatched-prior and distance-only controls. If generic controls match anatomy, describe regularization rather than anatomical reasoning.
4. **Uncertainty:** entropy-weighted versus regular graph, with matched supervision.
5. **Robustness:** compare clean-to-corrupted changes and covered-part rejection at matched visible recall. Synthetic masking is not a viewpoint benchmark.
6. **Reproducibility:** three seeds, paired intervals and per-part results. Do not promote a tiny single-seed advantage.

This suite does NOT implement published SOTA baselines, a second dataset, natural viewpoint annotations, diffusion counterfactuals, backbone finetuning, annotation-efficiency or a user study. Those are explicitly gated future stages in the research pipeline, not experiments claimed completed here.

Relevant prior work: [Zhu et al., CVPR 2025](https://openaccess.thecvf.com/content/CVPR2025/html/Zhu_Interpretable_Image_Classification_via_Non-parametric_Part_Prototype_Learning_CVPR_2025_paper.html), [Deformable ProtoPNet](https://arxiv.org/abs/2111.15000), [ProtoArgNet](https://ojs.aaai.org/index.php/AAAI/article/view/32173), [PIP-Net](https://openaccess.thecvf.com/content/CVPR2023/html/Nauta_PIP-Net_Patch-Based_Intuitive_Prototypes_for_Interpretable_Image_Classification_CVPR_2023_paper.html), [ProtoViT](https://proceedings.neurips.cc/paper_files/paper/2024/hash/48dfc849640344e2d58df0b5bb78c33b-Abstract-Conference.html). Spatial prototypes, visibility and abstention have prior art. This is a hypothesis-testing suite, not a claim of being first.


In [ ]:

# 15. Compact decision packet; preserve full Kaggle Output separately for reproducibility.
save_json(dict(protocol=PROTOCOL,run_key=RUN_KEY,versions=VERSIONS,
    input_notebook="dino-v2-v2.ipynb supplied by user",result_status="computed by this Kaggle run",
    caveat="CUB test is a reused development benchmark; external confirmation required"),OUT/"provenance.json")
with open(OUT/"environment.txt","w") as f:subprocess.run([sys.executable,"-m","pip","freeze"],stdout=f,check=False)
# Compact per-image arrays are sufficient for paired outcome/part analysis; full logits remain in Kaggle Output.
compact=OUT/"compact_predictions";compact.mkdir(exist_ok=True)
for seed in CFG["seeds"]:
    for name in RUN_VARIANTS:
        for policy in (["raw"] if name=="mean_linear" else ["raw",f"filtered_{PRIMARY_RECALL:.2f}"]):
            r=read_report(name,seed,policy)
            np.savez_compressed(compact/f"{name}_{seed}_{policy}.npz",**{k:v for k,v in r.items() if k!="logits"})
# A small automatically generated text overview, with no winner selected from test.
cols=["variant","policy","accuracy","pck","missing_emission_rate","visible_emission_recall","visibility_brier"]
overview=primary[cols].groupby(["variant","policy"]).mean(numeric_only=True).round(5)
(OUT/"READ_ME_FIRST.txt").write_text("CUB suite v3. All fractions, not percentages.\n"+overview.to_string()+
    "\n\nReview all_results.csv, paired_intervals.csv, tradeoff_curves.png and robustness_summary.csv.\n"
    "No automatic best model selection. These are development results.\n")
bundle=OUT/"decision_bundle.zip"
with zipfile.ZipFile(bundle,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=3) as archive:
    for p in sorted(OUT.rglob("*")):
        if not p.is_file() or p==bundle:continue
        rel=p.relative_to(OUT)
        if rel.parts[0] in {"cache","extracted_cub","corruption_cache"}:continue
        if p.suffix==".pt":continue
        if p.suffix==".npz" and rel.parts[0]!="compact_predictions":continue
        archive.write(p,str(rel))
print("Send back:",bundle,"MiB",round(bundle.stat().st_size/2**20,1))
print("Save the whole notebook Output to preserve feature caches, training banks, and .pt checkpoints.")
from IPython.display import FileLink
display(FileLink(str(bundle)))
